In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

BLEND_FILE = "blender/lod3.blend"
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"
RENDER_SCRIPT = "scripts/render_alzado.py"
RECTIFIED_DIR = "data/fachadas"
OUT_DIR = "renders"
ORIGIN = (440000.0, 4473700.0, 640.0)   # Local origin of the Blender model
RESOLUTION = 0.025                      # Meters per pixel, the same as the rectified photos

TOLEDO_LINE = [(439988.6, 4473740.7), (439984.4, 4473757.7), (439978.7, 4473780.4), (439979.4, 4473792.5),
               (439980.8, 4473819.3), (439982.0, 4473827.4), (439984.2, 4473842.1), (439985.5, 4473850.4)]

# Same windows as in 09_rectificar.ipynb (u_left > u_right, heights in meters above sea level):
JOBS = [
    {"name": "toledo_instituto_u41-80", "line": TOLEDO_LINE, "u_left": 80.5, "u_right": 40.5, "z_top": 662.5, "z_bottom": 644.0},
    {"name": "toledo_colegiata_u80-111", "line": TOLEDO_LINE, "u_left": 112.0, "u_right": 79.2, "z_top": 684.5, "z_bottom": 644.0},
]
ALPHA = 0.5   # Weight of the model in the overlay (0 = only the photo, 1 = only the model)

In [ ]:
# Celda 2
import os
import subprocess
import shapely
from shapely.geometry import LineString
from PIL import Image

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Celda 3
# Camera of each job: center of the window, and the normal of the facade toward the street.
# The street is on the left of the line direction: direction (dx, dy) -> normal (-dy, dx).
def camera_for(job):
    line = LineString(job["line"])
    u_mid = (job["u_left"] + job["u_right"]) / 2
    a = line.interpolate(u_mid - 1)
    b = line.interpolate(u_mid + 1)
    length = a.distance(b)
    dx, dy = (b.x - a.x) / length, (b.y - a.y) / length
    middle = line.interpolate(u_mid)
    x0, y0, z0 = ORIGIN
    center = (middle.x - x0, middle.y - y0, (job["z_top"] + job["z_bottom"]) / 2 - z0)
    return center, (-dy, dx)

In [ ]:
# Celda 4
# Render the elevation of each facade with Blender:
for job in JOBS:
    width_m = job["u_left"] - job["u_right"]
    height_m = job["z_top"] - job["z_bottom"]
    width_px = int(round(width_m / RESOLUTION))
    height_px = int(round(height_m / RESOLUTION))
    (cx, cy, cz), (nx, ny) = camera_for(job)
    out = OUT_DIR + "/alzado_" + job["name"] + ".png"
    command = [BLENDER, "-b", BLEND_FILE, "--python", RENDER_SCRIPT, "--", out,
               str(cx), str(cy), str(cz), str(nx), str(ny), str(width_m), str(height_m), str(width_px), str(height_px)]
    result = subprocess.run(command, capture_output=True, text=True)
    print(job["name"], "| return code", result.returncode, "|", width_px, "x", height_px, "->", out)

In [ ]:
# Celda 5
# Overlay: rectified photo and model elevation mixed half and half, and the two side by side:
for job in JOBS:
    photo = Image.open(RECTIFIED_DIR + "/" + job["name"] + ".jpg").convert("RGB")
    model = Image.open(OUT_DIR + "/alzado_" + job["name"] + ".png").convert("RGB").resize(photo.size)
    Image.blend(photo, model, ALPHA).save(OUT_DIR + "/comparacion_" + job["name"] + ".jpg", quality=90)
    pair = Image.new("RGB", (photo.width * 2, photo.height), "white")
    pair.paste(photo, (0, 0))
    pair.paste(model, (photo.width, 0))
    pair.save(OUT_DIR + "/lado_a_lado_" + job["name"] + ".jpg", quality=90)
    print("Saved:", OUT_DIR + "/comparacion_" + job["name"] + ".jpg")